#==================================================\n
# Code cell 1\n
#==================================================

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed\n
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python\n
# For example, here's several helpful packages to load\n
\n
import numpy as np # linear algebra\n
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)\n
\n
# Input data files are available in the read-only \"../input/\" directory\n
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory\n
\n
import os\n
for dirname, _, filenames in os.walk('/kaggle/input'):\n
    for filename in filenames:\n
        print(os.path.join(dirname, filename))\n
\n
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using \"Save & Run All\" \n
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

#==================================================\n
# Code cell 2\n
#==================================================

#==================================================\n
# Code cell 3\n
#==================================================

In [ ]:
input = '/kaggle/input/source/'\n
output = '/kaggle/working/'

#==================================================\n
# Code cell 4\n
#==================================================

In [ ]:
## 基础工具\n
import numpy as np\n
import pandas as pd\n
import warnings\n
import matplotlib\n
import matplotlib.pyplot as plt\n
import seaborn as sns\n
from scipy.special import jn\n
from IPython.display import display, clear_output\n
import time\n
\n
warnings.filterwarnings('ignore')\n
matplotlib.rcParams['font.sans-serif']=[u'simHei']\n
%matplotlib inline\n
\n
## 数据处理的\n
import pandas_profiling as pp\n
#缺失值可视化工具\n
import missingno as msno\n
\n
## 模型预测的\n
import sklearn\n
from sklearn import linear_model\n
from sklearn import preprocessing\n
from sklearn.svm import SVR\n
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor\n
from sklearn.tree import DecisionTreeClassifier\n
\n
## 数据降维处理的\n
from sklearn.decomposition import PCA,FastICA,FactorAnalysis,SparsePCA\n
\n
## 处理数据不平衡\n
from imblearn.over_sampling import SMOTE\n
from imblearn.under_sampling import RandomUnderSampler\n
from imblearn.pipeline import Pipeline as ImbPipeline\n
\n
import lightgbm as lgb\n
import xgboost as xgb\n
\n
## 参数搜索和评价的\n
from sklearn.model_selection import GridSearchCV,cross_val_score,StratifiedKFold,train_test_split\n
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, classification_report\n
from sklearn.pipeline import Pipeline\n

#==================================================\n
# Code cell 5\n
#==================================================

In [ ]:
#数据加载\n
dataset=pd.read_csv(input+'SupplyChain.csv', encoding='unicode_escape')\n
dataset

#==================================================\n
# Code cell 6\n
#==================================================

In [ ]:
data = dataset.copy()

#==================================================\n
# Code cell 7\n
#==================================================

In [ ]:
# 18万比订单，53个特征\n
print(data.shape)\n
temp = data.isnull().sum()\n
temp[temp>0]

#==================================================\n
# Code cell 8\n
#==================================================

In [ ]:
data['Customer Lname'].value_counts() #Smith        64104\n
data['Customer Lname'].fillna(data['Customer Lname'].mode()[0], inplace=True)

#==================================================\n
# Code cell 9\n
#==================================================

In [ ]:
data['Customer Zipcode'].value_counts()\n
data['Customer Zipcode'].fillna(data['Customer Zipcode'].mode()[0], inplace=True)

#==================================================\n
# Code cell 10\n
#==================================================

In [ ]:
data.select_dtypes(exclude=[object]).columns 

#==================================================\n
# Code cell 11\n
#==================================================

In [ ]:
#将Firs tName 与LastName进行合并=>Full Name \n
data['Customer Full Name'] = data['Customer Fname'] + data['Customer Lname']\n
data[['Customer Full Name', 'Customer Fname', 'Customer Lname']]

#==================================================\n
# Code cell 12\n
#==================================================

In [ ]:
# df = dataset['Customer State'].astype('category').copy()  # Categorize!\n
# df\n
\n
# 选择需要处理的object columns\n
# dataset.select_dtypes(include=[object]).columns  \n
# dataset.select_dtypes(exclude=[object]).columns  \n
\n
# 转换格式 data['Customer State'].astype('category')\n
astype_columns = ['Type', 'Delivery Status', 'Category Name', 'Customer City', 'Customer Country', 'Customer Fname', 'Customer Lname', \n
                  'Customer Segment', 'Customer State', 'Customer Street', 'Department Name', 'Market', 'Customer Full Name',\n
                  'Order City', 'Order Country', 'Order Region', 'Order State',\n
                  'Product Name', 'Shipping Mode', 'shipping date (DateOrders)', 'order date (DateOrders)', 'Order Status'\n
                 ]\n
\n
# drop\n
drop_columns = ['Customer Email', 'Customer Password', 'Product Image',  'Order Zipcode', 'Product Description']\n
\n
# 特征梳理\n
# 时间日期多维度： order date (DateOrders)   shipping date (DateOrders)\n
# 补全缺失值后：Customer Fname + Customer Lname = Customer Full Name\n
feature_columns = ['order date (DateOrders)', 'shipping date (DateOrders)']\n
\n
# y值\n
y_column = ['Order Status']

#==================================================\n
# Code cell 13\n
#==================================================

In [ ]:
data.drop(drop_columns, axis=1, inplace=True)\n
data.info()

#==================================================\n
# Code cell 14\n
#==================================================

In [ ]:
#  order date (DateOrders)\n
#按照不同的时间维度（年，月，星期，小时）的趋势\n
#data[['order date (DateOrders)']]\n
#创建时间影索引\n
temp = pd.DatetimeIndex(data['order date (DateOrders)'])\n
temp

#==================================================\n
# Code cell 15\n
#==================================================

In [ ]:
# order date (DateOrders) 字段中的时间多尺度 year, month, weekday, hour, month_year\n
data['order_year'] = temp.year\n
data['order_month'] = temp.month\n
data['order_week_day'] = temp.weekday\n
data['order_hour'] = temp.hour\n
#data['order_month_year'] = temp.to_period('M')  auto-sklearn unsported\n
data

#==================================================\n
# Code cell 16\n
#==================================================

In [ ]:
# 对销售额进行探索，按照不同的时间维度（年，月，星期，小时）的趋势\n
plt.subplot(4, 2, 1)\n
df_year = data.groupby('order_year')\n
df_year['Sales'].mean().plot(figsize=(12, 12), title='Mean sales in Years')\n
\n
plt.subplot(4, 2, 2)\n
df_day = data.groupby('order_week_day')\n
df_day['Sales'].mean().plot(figsize=(12,12), title='Average sales in days')\n
\n
plt.subplot(4, 2, 3)\n
df_hour = data.groupby('order_hour')\n
df_hour['Sales'].mean().plot(figsize=(12,12), title='Average sales in Hours')\n
\n
plt.subplot(4, 2, 4)\n
df_month = data.groupby('order_month')\n
df_month['Sales'].mean().plot(figsize=(12, 12), title='Average sales in Months')

#==================================================\n
# Code cell 17\n
#==================================================

In [ ]:
#  shipping date (DateOrders)\n
#按照不同的时间维度（年，月，星期，小时）的趋势\n
#data[['shipping date (DateOrders)']]\n
#创建时间影索引\n
temp = pd.DatetimeIndex(data['shipping date (DateOrders)'])\n
temp

#==================================================\n
# Code cell 18\n
#==================================================

In [ ]:
# shipping date (DateOrders) 字段中的时间多尺度 year, month, weekday, hour, month_year\n
data['shipping_year'] = temp.year\n
data['shipping_month'] = temp.month\n
data['shipping_week_day'] = temp.weekday\n
data['shipping_hour'] = temp.hour\n
# data['shipping_month_year'] = temp.to_period('M')    auto-sklearn unsported\n
data

#==================================================\n
# Code cell 19\n
#==================================================

In [ ]:
for column in astype_columns:\n
    data[column] = data[column].astype('category')

#==================================================\n
# Code cell 20\n
#==================================================

In [ ]:
# 18万比订单，53个特征\n
print(data.shape)\n
temp = data.isnull().sum()\n
temp[temp>0]

#==================================================\n
# Code cell 21\n
#==================================================

In [ ]:
# data.select_dtypes(include=[object]).columns  \n
# data.select_dtypes(exclude=[object]).columns  \n
data.info()\n
data.select_dtypes(include=[object]).columns 

#==================================================\n
# Code cell 22\n
#==================================================

In [ ]:
#  批量Labels encoding:\n
\n
# preprocessing.LabelBinarizer\n
# preprocessing.LabelEncoder\n
\n
data2 = data.copy()\n
\n
str_cols = data2.select_dtypes(include=['category']).columns\n
clfs = {c:preprocessing.LabelEncoder() for c in str_cols}\n
\n
for col, clf in clfs.items():\n
    data2[col] = clfs[col].fit_transform(data2[col])\n
\n
data2\n
\n
# 标签反转演示\n
# for col, clf in clfs.items():\n
#     display(col, clfs[col].inverse_transform([0]))

#==================================================\n
# Code cell 23\n
#==================================================

In [ ]:
# 标签反转演示\n
display(\"Order Status\", clfs[\"Order Status\"].inverse_transform([0,1,2,3,4,5,6,7,8]))\n
\n
display(\"Order Status\", clfs[\"Order Status\"].inverse_transform([8]))

#==================================================\n
# Code cell 25\n
#==================================================

In [ ]:
display(data['Order Status'].value_counts())\n
data['Order Status'].value_counts().plot.bar()

#==================================================\n
# Code cell 26\n
#==================================================

In [ ]:
#  切分训练集、测试集\n
\n
# # ### 删除不适用的特征\n
drop_features = ['Order Status', \n
                 #'shipping_year', 'shipping_month', 'shipping_week_day', 'shipping_hour',\n
                 #'order_year', 'order_month', 'order_week_day', 'order_hour',\n
                 'order date (DateOrders)', 'shipping date (DateOrders)'\n
                ]\n
feature_cols = [column for column in data2.columns if column not in drop_features]\n
# display(feature_cols)\n
\n
astype_features = [\n
                   'shipping_year', 'shipping_month', 'shipping_week_day', 'shipping_hour',\n
                   'order_year', 'order_month', 'order_week_day', 'order_hour'\n
                  ]\n
\n
for column in astype_features:\n
    data[column] = data[column].astype('category')\n
\n
# # ###随机采样，测试程序用，设置一个小样本的数据集，快速预览模型\n
# np.random.seed(10)\n
# \n
# #按照百分比抽样，不放回\n
# data2_sample = data2.sample(frac=0.01) #抽取20%的数据\n
# display(data2_sample.shape)\n
# \n
# # 样本数据\n
# X = data2_sample[feature_cols]\n
# y = data2_sample['Order Status']\n
# \n
# 全量特征\n
# AC 0.315976069133614\n
# CPU times: user 3min 24s, sys: 1min 20s, total: 4min 44s\n
# Wall time: 1min 12s\n
\n
\n
\n
# 全量数据\n
X = data2[feature_cols]\n
y = data2['Order Status']\n
\n
# 转换为二分类，需要修改模型\n
# \"SUSPECTED_FRAUD\" --> 8\n
# y_2 = y.apply(lambda x : 1 if x == 8 else 0).copy()\n
\n
# 使用 stratify 保证训练集和测试集中类别比例一致\n
X_train, X_test, y_train, y_test = \\\n
        train_test_split(X, y, random_state=2021, stratify=y)\n
\n
# 使用 SMOTE 对训练集进行过采样，平衡类别\n
print(\"原始训练集类别分布:\")\n
print(pd.Series(y_train).value_counts())\n
\n
smote = SMOTE(random_state=2021, k_neighbors=5)\n
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)\n
\n
print(\"\\nSMOTE后训练集类别分布:\")\n
print(pd.Series(y_train_resampled).value_counts())

#==================================================\n
# Code cell 27\n
#==================================================

In [ ]:
%%time\n
\n
# GaussianNB\n
from sklearn.naive_bayes import GaussianNB\n
\n
gnb = GaussianNB()\n
# 使用平衡后的数据进行训练\n
y_pred = gnb.fit(X_train_resampled, y_train_resampled).predict(X_test)\n
print(\"Number of mislabeled points out of a total %d points : %d\"\n
      % (X_test.shape[0], (y_test != y_pred).sum()))\n
\n
# 转换为二分类标签\n
display( pd.Series(y_test).value_counts() )\n
y_test_2 = y_test.apply(lambda x : 1 if x ==8 else 0).copy()\n
y_test_2.value_counts()\n
\n
# # 转换为二分类标签\n
display( pd.Series(y_pred).value_counts() )\n
y_pred_2 = pd.Series(y_pred).apply(lambda x : 1 if x ==8 else 0).copy()\n
pd.Series(y_pred_2).value_counts()\n
\n
# ===== 详细评估指标 =====\n
print('\n========== LinearSVC 模型评估 ==========')\n
\n
# 混淆矩阵\n
m = confusion_matrix(y_test_2, y_pred_2)\n
print('\n混淆矩阵：')\n
print(m)\n
\n
# 准确率\n
print(f\"\\n准确率 (Accuracy): {accuracy_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 精确率、召回率、F1分数（针对欺诈类别）\n
print(f\"精确率 (Precision): {precision_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"召回率 (Recall): {recall_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"F1分数 (F1-Score): {f1_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 分类报告\n
print('\n分类报告：')\n
print(classification_report(y_test_2, y_pred_2, target_names=['正常订单', '欺诈订单']))

#==================================================\n
# Code cell 28\n
#==================================================

In [ ]:
%%time\n
\n
# LinearSVC\n
from sklearn.svm import LinearSVC\n
from sklearn.pipeline import make_pipeline\n
from sklearn.preprocessing import StandardScaler\n
# 添加 class_weight='balanced' 处理不平衡数据\n
clf = make_pipeline(StandardScaler(),\n
                    LinearSVC(random_state=0, tol=1e-5, class_weight='balanced'))\n
clf.fit(X_train_resampled, y_train_resampled)\n
\n
# print(clf.named_steps['linearsvc'].coef_)\n
# print(clf.named_steps['linearsvc'].intercept_)\n
\n
y_pred = clf.predict(X_test)\n
\n
# 转换为二分类标签\n
display( pd.Series(y_test).value_counts() )\n
y_test_2 = y_test.apply(lambda x : 1 if x ==8 else 0).copy()\n
y_test_2.value_counts()\n
\n
# # 转换为二分类标签\n
display( pd.Series(y_pred).value_counts() )\n
y_pred_2 = pd.Series(y_pred).apply(lambda x : 1 if x ==8 else 0).copy()\n
pd.Series(y_pred_2).value_counts()\n
\n
# ===== 详细评估指标 =====\n
print('\n========== KNeighborsClassifier 模型评估 ==========')\n
\n
# 混淆矩阵\n
m = confusion_matrix(y_test_2, y_pred_2)\n
print('\n混淆矩阵：')\n
print(m)\n
\n
# 准确率\n
print(f\"\\n准确率 (Accuracy): {accuracy_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 精确率、召回率、F1分数（针对欺诈类别）\n
print(f\"精确率 (Precision): {precision_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"召回率 (Recall): {recall_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"F1分数 (F1-Score): {f1_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 分类报告\n
print('\n分类报告：')\n
print(classification_report(y_test_2, y_pred_2, target_names=['正常订单', '欺诈订单']))

#==================================================\n
# Code cell 29\n
#==================================================

In [ ]:
%%time\n
\n
# KNeighborsClassifier\n
from sklearn.neighbors import KNeighborsClassifier\n
# KNN 使用平衡后的数据\n
neigh = KNeighborsClassifier(n_neighbors=3)\n
neigh.fit(X_train_resampled, y_train_resampled)\n
      \n
y_pred = neigh.predict(X_test)\n
\n
# 转换为二分类标签\n
display( pd.Series(y_test).value_counts() )\n
y_test_2 = y_test.apply(lambda x : 1 if x ==8 else 0).copy()\n
y_test_2.value_counts()\n
\n
# # 转换为二分类标签\n
display( pd.Series(y_pred).value_counts() )\n
y_pred_2 = pd.Series(y_pred).apply(lambda x : 1 if x ==8 else 0).copy()\n
pd.Series(y_pred_2).value_counts()\n
\n
# ===== 详细评估指标 =====\n
print('\n========== LinearDiscriminantAnalysis 模型评估 ==========')\n
\n
# 混淆矩阵\n
m = confusion_matrix(y_test_2, y_pred_2)\n
print('\n混淆矩阵：')\n
print(m)\n
\n
# 准确率\n
print(f\"\\n准确率 (Accuracy): {accuracy_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 精确率、召回率、F1分数（针对欺诈类别）\n
print(f\"精确率 (Precision): {precision_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"召回率 (Recall): {recall_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"F1分数 (F1-Score): {f1_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 分类报告\n
print('\n分类报告：')\n
print(classification_report(y_test_2, y_pred_2, target_names=['正常订单', '欺诈订单']))

#==================================================\n
# Code cell 30\n
#==================================================

In [ ]:
%%time \n
\n
# LinearDiscriminantAnalysis\n
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis\n
\n
clf = LinearDiscriminantAnalysis()\n
# 使用平衡后的数据\n
clf.fit(X_train_resampled, y_train_resampled)\n
\n
y_pred = clf.predict(X_test)\n
\n
# 转换为二分类标签\n
display( pd.Series(y_test).value_counts() )\n
y_test_2 = y_test.apply(lambda x : 1 if x ==8 else 0).copy()\n
y_test_2.value_counts()\n
\n
# # 转换为二分类标签\n
display( pd.Series(y_pred).value_counts() )\n
y_pred_2 = pd.Series(y_pred).apply(lambda x : 1 if x ==8 else 0).copy()\n
pd.Series(y_pred_2).value_counts()\n
\n
# ===== 详细评估指标 =====\n
print('\n========== DecisionTreeClassifier 模型评估 ==========')\n
\n
# 混淆矩阵\n
m = confusion_matrix(y_test_2, y_pred_2)\n
print('\n混淆矩阵：')\n
print(m)\n
\n
# 准确率\n
print(f\"\\n准确率 (Accuracy): {accuracy_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 精确率、召回率、F1分数（针对欺诈类别）\n
print(f\"精确率 (Precision): {precision_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"召回率 (Recall): {recall_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"F1分数 (F1-Score): {f1_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 分类报告\n
print('\n分类报告：')\n
print(classification_report(y_test_2, y_pred_2, target_names=['正常订单', '欺诈订单']))

#==================================================\n
# Code cell 31\n
#==================================================

In [ ]:
%%time\n
\n
# DecisionTreeClassifier\n
#from sklearn.model_selection import cross_val_score\n
from sklearn.tree import DecisionTreeClassifier\n
# 添加 class_weight='balanced' 处理不平衡数据\n
clf = DecisionTreeClassifier(random_state=2021, class_weight='balanced')\n
\n
clf.fit(X_train_resampled, y_train_resampled)\n
y_pred = clf.predict(X_test)\n
\n
# 转换为二分类标签\n
display( pd.Series(y_test).value_counts() )\n
y_test_2 = y_test.apply(lambda x : 1 if x ==8 else 0).copy()\n
y_test_2.value_counts()\n
\n
# # 转换为二分类标签\n
display( pd.Series(y_pred).value_counts() )\n
y_pred_2 = pd.Series(y_pred).apply(lambda x : 1 if x ==8 else 0).copy()\n
pd.Series(y_pred_2).value_counts()\n
\n
# ===== 详细评估指标 =====\n
print('\n========== RandomForestClassifier 模型评估 ==========')\n
\n
# 混淆矩阵\n
m = confusion_matrix(y_test_2, y_pred_2)\n
print('\n混淆矩阵：')\n
print(m)\n
\n
# 准确率\n
print(f\"\\n准确率 (Accuracy): {accuracy_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 精确率、召回率、F1分数（针对欺诈类别）\n
print(f\"精确率 (Precision): {precision_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"召回率 (Recall): {recall_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"F1分数 (F1-Score): {f1_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 分类报告\n
print('\n分类报告：')\n
print(classification_report(y_test_2, y_pred_2, target_names=['正常订单', '欺诈订单']))

#==================================================\n
# Code cell 32\n
#==================================================

In [ ]:
%%time\n
\n
# RandomForestClassifier\n
\n
from sklearn.ensemble import RandomForestClassifier\n
# 添加 class_weight='balanced' 处理不平衡数据\n
clf = RandomForestClassifier(max_depth=7, random_state=2021, class_weight='balanced')\n
clf.fit(X_train_resampled, y_train_resampled)\n
\n
y_pred = clf.predict(X_test)\n
\n
# 转换为二分类标签\n
display( pd.Series(y_test).value_counts() )\n
y_test_2 = y_test.apply(lambda x : 1 if x ==8 else 0).copy()\n
y_test_2.value_counts()\n
\n
# # 转换为二分类标签\n
display( pd.Series(y_pred).value_counts() )\n
y_pred_2 = pd.Series(y_pred).apply(lambda x : 1 if x ==8 else 0).copy()\n
pd.Series(y_pred_2).value_counts()\n
\n
# 混淆矩阵\n
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report\n
m = confusion_matrix(y_test_2, y_pred_2)\n
print('\n混淆矩阵：')\n
print(m)\n
\n
# 准确率\n
print(f\"\\n准确率 (Accuracy): {accuracy_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 精确率、召回率、F1分数（针对欺诈类别）\n
print(f\"精确率 (Precision): {precision_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"召回率 (Recall): {recall_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"F1分数 (F1-Score): {f1_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 分类报告\n
print('\n分类报告：')\n
print(classification_report(y_test_2, y_pred_2, target_names=['正常订单', '欺诈订单']))

#==================================================\n
# Code cell 33\n
#==================================================

In [ ]:
%%time\n
\n
#XGBClassifier \n
# 计算类别权重比例用于 XGBoost\n
# 针对多分类问题，使用平衡后的数据或保持原数据但不设置scale_pos_weight\n
xgr = xgb.XGBClassifier(learning_rate=0.1,\n
                        n_estimators=1000,         # 树的个数--1000棵树建立xgboost\n
                        max_depth=6,               # 树的深度\n
                        min_child_weight = 1,      # 叶子节点最小权重\n
                        gamma=0.,                  # 惩罚项中叶子结点个数前的参数\n
                        subsample=0.8,             # 随机选择80%样本建立决策树\n
                        colsample_btree=0.8,       # 随机选择80%特征建立决策树\n
                        objective='multi:softmax', # 指定损失函数\n
                        random_state=27            # 随机数\n
                        )\n
\n
# 使用 SMOTE 平衡后的数据训练\n
xgr.fit(X_train_resampled, y_train_resampled)\n
y_pred = xgr.predict(X_test)\n
\n
### plot feature importance\n
fig,ax = plt.subplots(figsize=(15,15))\n
xgb.plot_importance(xgr,\n
                height=0.5,\n
                ax=ax,\n
                max_num_features=64)\n
plt.show()\n
\n
# 转换为二分类标签\n
display( pd.Series(y_test).value_counts() )\n
y_test_2 = y_test.apply(lambda x : 1 if x ==8 else 0).copy()\n
y_test_2.value_counts()\n
\n
# # 转换为二分类标签\n
display( pd.Series(y_pred).value_counts() )\n
y_pred_2 = pd.Series(y_pred).apply(lambda x : 1 if x ==8 else 0).copy()\n
pd.Series(y_pred_2).value_counts()\n
\n
# 混淆矩阵\n
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report\n
m = confusion_matrix(y_test_2, y_pred_2)\n
print('\n混淆矩阵：')\n
print(m)\n
\n
# 准确率\n
print(f\"\\n准确率 (Accuracy): {accuracy_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 精确率、召回率、F1分数（针对欺诈类别）\n
print(f\"精确率 (Precision): {precision_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"召回率 (Recall): {recall_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"F1分数 (F1-Score): {f1_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 分类报告\n
print('\n分类报告：')\n
print(classification_report(y_test_2, y_pred_2, target_names=['正常订单', '欺诈订单']))

#==================================================\n
# Code cell 34\n
#==================================================

In [ ]:
# %%time\n
\n
# # 训练\n
# LR = sklearn.linear_model.LinearRegression()  #报错\n
# LR = sklearn.linear_model.LogisticRegression(multi_class=\"multinomial\", solver=\"newton-cg\", max_iter=1000)\n
\n
# 多分类，添加 class_weight='balanced' 处理不平衡数据\n
LR = sklearn.linear_model.LogisticRegression(multi_class=\"multinomial\", solver=\"newton-cg\", max_iter=1000, class_weight='balanced') \n
\n
reg = LR.fit(X_train_resampled, y_train_resampled)\n
reg.score(X_train, y_train)\n
reg.coef_\n
reg.intercept_\n
y_pred = reg.predict(X_test)\n
\n
print(\"LR\")\n
print(classification_report(y_test, y_pred))\n
print(\"AC\",accuracy_score(y_test, y_pred))\n
\n
\n
# 转换为二分类标签\n
display( pd.Series(y_test).value_counts() )\n
y_test_2 = y_test.apply(lambda x : 1 if x ==8 else 0).copy()\n
y_test_2.value_counts()\n
\n
# # 转换为二分类标签\n
display( pd.Series(y_pred).value_counts() )\n
y_pred_2 = pd.Series(y_pred).apply(lambda x : 1 if x ==8 else 0).copy()\n
pd.Series(y_pred_2).value_counts()\n
\n
# 混淆矩阵\n
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report\n
m = confusion_matrix(y_test_2, y_pred_2)\n
print('\n混淆矩阵：')\n
print(m)\n
\n
# 准确率\n
print(f\"\\n准确率 (Accuracy): {accuracy_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 精确率、召回率、F1分数（针对欺诈类别）\n
print(f\"精确率 (Precision): {precision_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"召回率 (Recall): {recall_score(y_test_2, y_pred_2):.4f}\")\n
print(f\"F1分数 (F1-Score): {f1_score(y_test_2, y_pred_2):.4f}\")\n
\n
# 分类报告\n
print('\n分类报告：')\n
print(classification_report(y_test_2, y_pred_2, target_names=['正常订单', '欺诈订单']))

#==================================================\n
# Code cell 35\n
#==================================================

In [ ]:
#  模型调优\n
# 交叉验证，\n
# 网格搜索

#==================================================\n
# Code cell 37\n
#==================================================